In [1]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 08:38:50.217513: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 08:38:50.804291: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
  "partitions": 1,
  "num_of_workers": 1,
  "iterations": 2,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 08:38:53,153 [DEBUG] [Rain] Rain is initialized
2023-07-04 08:38:53,154 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 08:38:53,155 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:38:53,155 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-04 08:38:53,157 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 08:38:53,158 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:38:53,159 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 08:38:53,160 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:38:53,161 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:38:53,162 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 08:38:53,168 [DEBUG] [Rain] Creating workers
2023-07-04 08:38:53,173 [INFO] [Provisioner] provisioner is serving
2023-07-04 08:38:53,174 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 08:38:53,176 [INFO] [Coordinator] coordinator is serving
2023-07-04 08:38:53,176 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 08:38:53,180 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:38:53,181 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 08:38:53,183 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 08:38:53,184 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:38:53,186 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:38:53,186 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:38:53,188 [INFO] [W

469/469 [==============================] - 2s 3ms/step - loss: 0.4249 - accuracy: 0.8694
sending data to coordinator


2023-07-04 08:39:08,884 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:39:08,885 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 08:39:08,954 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:39:08,961 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-04 08:39:08,985 [DEBUG] [DeepLearning] Iteration 1/2 complete for worker 1.
2023-07-04 08:39:08,986 [DEBUG] [DeepLearning] Starting iteration 2/2
2023-07-04 08:39:08,986 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-04 08:39:08,987 [DEBUG] [DividerAmbassador] Sending ../../../..//RainData/divider/1.pkl to worker1
2023-07-04 08:39:09,057 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 08:39:09,058 [DEBUG] [DividerAmbassador] divide

469/469 [==============================] - 2s 3ms/step - loss: 0.1959 - accuracy: 0.9415
sending data to coordinator


2023-07-04 08:39:11,103 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 08:39:11,104 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 08:39:11,175 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:39:11,182 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
DEBUG:DeepLearning:Asynchronous update is done by worker 1
2023-07-04 08:39:11,201 [DEBUG] [DeepLearning] Iteration 2/2 complete for worker 1.
DEBUG:DeepLearning:Iteration 2/2 complete for worker 1.
2023-07-04 08:39

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1062 - accuracy: 0.9668

Test accuracy: 96.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 08:39:11,518 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-04 08:39:11,521 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-04 08:39:11,522 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-04 08:39:11,524 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-04 08:39:11,525 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-04 08:39:11,526 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:39:11,528 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-04 08:39:11,

469/469 [==============================] - 2s 4ms/step - loss: 0.1504 - accuracy: 0.9542


2023-07-04 08:39:33,396 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 08:39:33,398 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to coordinator


2023-07-04 08:39:33,601 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-04 08:39:33,638 [DEBUG] [DeepLearning] Iteration 1/2 complete.
DEBUG:DeepLearning:Iteration 1/2 complete.
2023-07-04 08:39:33,641 [DEBUG] [DeepLearning] Starting iteration 2/2
DEBUG:DeepLearning:Starting iteration 2/2
2023-07-04 08:39:33,671 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
DEBUG:DividerAmbassador:127.0.0.1:50151
2023-07-04 08:39:33,673 [DEBUG] [DividerAmbassador] Sending ../../../..//RainData/divider/1.pkl to worker1
DEBUG:DividerAmbassador:Sending ../../../..//RainData/divider/1.pkl to worker1
2023-07-04 08:39:33,780 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
DEBUG:DividerAmbassador:divider received: File downloaded successfully after sending the model to worke

469/469 [==============================] - 3s 4ms/step - loss: 0.1335 - accuracy: 0.9607


2023-07-04 08:39:36,719 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 08:39:36,720 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_2_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_2_trained.pkl from worker1


sending data to coordinator


2023-07-04 08:39:36,809 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_2_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/1_2_trained.pkl from worker1 successfully
2023-07-04 08:39:36,838 [DEBUG] [DeepLearning] Iteration 2/2 complete.
DEBUG:DeepLearning:Iteration 2/2 complete.
2023-07-04 08:39:36,840 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-04 08:39:36,842 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving
2023-07-04 08:39:36,844 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-04 08:39:36,845 [DEBUG] [Divider] Divider stopped serving
DEBUG:Divider:Divider stopped serving
2023-07-04 08:39:36,847 [INFO] [Provisioner] provisioner stopped serving
INFO:Provisioner:provisioner stopped serving


In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.0840 - accuracy: 0.9754

Test accuracy: 97.5%
